# GAR on Sudoku-Extreme

GAR and its continuation variant (*continue instead of restart*) run for up to 64 retries on the 1,000 evaluation
puzzles, with three seeds, and are compared under the same harness with TRM, whose retries continue without
perturbation (depth alone), and with PTRM. PTRM's rollouts are read from the cache written by
`SudokuExtreme_continue_and_PTRM`, so run that notebook first.

Set `CKPT_TRM` and `CACHE_DIR` below. Each pass is saved in `CACHE_DIR` and reloaded when the notebook runs again.

In [ ]:
!pip -q install einops pydantic argdantic tqdm huggingface_hub
!test -d TRM || git clone -q https://github.com/SamsungSAILMontreal/TinyRecursiveModels TRM
!git -C TRM checkout -q c011037

In [ ]:
import os, sys, json, subprocess, dataclasses

import numpy as np
import pandas as pd
import torch
import yaml

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

CKPT_TRM = "/content/drive/MyDrive/GAR/sudoku/checkpoints/step_65100"  # with all_config.yaml next to it
CACHE_DIR = "/content/drive/MyDrive/GAR/sudoku/cache"
DATA_PATH = "data/sudoku-extreme-1k-aug-1000"
GLOBAL_BATCH_SIZE = 768
DEVICE = "cuda"

sys.path.insert(0, os.path.abspath("TRM"))
from models.losses import ACTLossHead, IGNORE_LABEL_ID
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1
from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig

os.makedirs(CACHE_DIR, exist_ok=True)
pd.options.display.float_format = "{:.2f}".format

## Checkpoint and test split

In [ ]:
if not os.path.exists(os.path.join(DATA_PATH, "test", "dataset.json")):
    subprocess.run([sys.executable, "TRM/dataset/build_sudoku_dataset.py", "--output-dir", DATA_PATH,
                    "--subsample-size", "1000", "--num-aug", "1000"], check=True)
test_set = PuzzleDataset(PuzzleDatasetConfig(seed=0, dataset_paths=[DATA_PATH], global_batch_size=GLOBAL_BATCH_SIZE,
                                             test_set_mode=True, epochs_per_iter=1, rank=0, num_replicas=1),
                         split="test")
test_set._lazy_load_dataset()
TEST = test_set._data["all"]

with open(os.path.join(os.path.dirname(CKPT_TRM), "all_config.yaml")) as f:
    arch = yaml.safe_load(f)["arch"]
loss_type = arch.pop("loss")["loss_type"]
del arch["name"]
state = {k.replace("_orig_mod.", "", 1): v for k, v in torch.load(CKPT_TRM, map_location=DEVICE).items()}
config = dict(arch, batch_size=GLOBAL_BATCH_SIZE, vocab_size=test_set.metadata.vocab_size,
              seq_len=test_set.metadata.seq_len,
              num_puzzle_identifiers=state["model.inner.puzzle_emb.weights"].shape[0])
with torch.device(DEVICE):
    model = ACTLossHead(TinyRecursiveReasoningModel_ACTV1(config), loss_type=loss_type)
model.load_state_dict(state, strict=False, assign=True)
model.eval()  # halting is off: every block runs exactly T supervision steps
inner = model.model.inner
T = model.model.config.halt_max_steps  # supervision steps per block (16)
P = inner.puzzle_emb_len  # prefix positions before the 81 cells (16)

## The harness

The standard pass runs one block of `T` supervision steps. The gate re-opens a puzzle whose halting logit `q` is
negative, and a retry runs one more block on it. The search stops at the first block with `q >= 0`, or after
`ROUNDS` retries, and returns the grid of highest `q` among the blocks run (the fallback). GAR (`restart`) and its
continuation variant (`randvec`) resample the fillable cells of `y` (`z_H` in TRM's code) at each retry, with the same
random directions: GAR first returns `y` and `z` to their initial states, its variant continues the failed trajectory.
TRM (`none`) continues the failed trajectory and changes nothing.

In [ ]:
Q_THR = 0.0  # the gate re-opens a puzzle when q < Q_THR, and a block with q >= Q_THR stops the search
BLANK = 1  # input token of the fillable cells
ROUNDS = 64
SEEDS = [0, 1, 2]


def clone(x):
    """Deep copy of a TRM carry."""
    if torch.is_tensor(x):
        return x.clone()
    if dataclasses.is_dataclass(x):
        return type(x)(**{f.name: clone(getattr(x, f.name)) for f in dataclasses.fields(x)})
    if isinstance(x, dict):
        return {k: clone(v) for k, v in x.items()}
    return x


def correct(pred, batch):
    """Exact match on the labelled cells, as counted by TRM's evaluator, and the rows that have labels."""
    mask = batch["labels"] != IGNORE_LABEL_ID
    return (mask & (pred == batch["labels"])).sum(-1) == mask.sum(-1), mask.any(-1)


@torch.no_grad()
def run_block(batch, carry=None):
    """One block of T supervision steps; carry=None starts from the initial states (the standard pass)."""
    if carry is None:
        with torch.device(DEVICE):
            carry = model.initial_carry(batch)
    for _ in range(T):
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
    return carry, out["preds"], out["q_halt_logits"].float()


@torch.no_grad()
def retry(carry, fill, active, arm, gen):
    """Prepares a retry of the rows `active`: resamples the fillable cells of y with isotropic directions at the
    norm of y_init, after returning y and z to their initial states on every position for a restart. The arm `none`
    changes nothing."""
    zH, zL = carry.inner_carry.z_H, carry.inner_carry.z_L
    if arm != "none":
        e = torch.randn(zH[:, P:].shape, generator=gen, device=zH.device, dtype=torch.float32)
        e = e / e.norm(dim=-1, keepdim=True).clamp_min(1e-9) * inner.H_init.float().norm()
        if arm == "restart":
            rows = active.view(-1, 1, 1)
            zH.copy_(torch.where(rows, inner.H_init.to(zH.dtype), zH))
            zL.copy_(torch.where(rows, inner.L_init.to(zL.dtype), zL))
        cells = (fill & active.view(-1, 1)).unsqueeze(-1)
        zH[:, P:] = torch.where(cells, e.to(zH.dtype), zH[:, P:])
    carry.halted.fill_(False)  # so that the next step continues from this state instead of resetting it
    carry.steps.zero_()


@torch.no_grad()
def run_pass(arm, seed):
    """The harness on the pool with retries of `arm`. Per puzzle, in pool order: the standard pass (ok0, q0, pred0),
    ok[:, r] (the returned grid is correct after r retries) and act[:, r] (the puzzle is open before retry r + 1)."""
    res = {"ok0": [], "q0": [], "pred0": [], "ok": [], "act": []}
    for b, start in enumerate(range(0, POOL_N, GLOBAL_BATCH_SIZE)):
        batch = pool_batch(POOL_IDX[start:start + GLOBAL_BATCH_SIZE])
        fill = batch["inputs"] == BLANK
        gen = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b)
        carry, pred0, q0 = run_block(batch)
        ok0, valid = correct(pred0, batch)
        ok0 &= valid
        best_p, best_q = pred0.clone(), q0.clone()
        active = valid & (q0 < Q_THR)
        ok, act = [ok0], []
        for _ in range(ROUNDS):
            act.append(active.clone())
            if active.any():
                retry(carry, fill, active, arm, gen)
                carry, p, q = run_block(batch, carry)
                better = active & (q > best_q)
                best_p = torch.where(better.view(-1, 1), p, best_p)
                best_q = torch.where(better, q, best_q)
                active &= q < Q_THR
            ok.append(correct(best_p, batch)[0] & valid)
        rows = valid.nonzero(as_tuple=True)[0]  # drops the padding rows of the last batch
        for k, v in (("ok0", ok0), ("q0", q0), ("pred0", pred0), ("ok", torch.stack(ok, 1)),
                     ("act", torch.stack(act, 1))):
            res[k].append(v[rows].cpu().numpy())
    return {k: np.concatenate(v) for k, v in res.items()}

## The 1,000 evaluation puzzles and the cache

In [ ]:
POOL_N = 1000
POOL_IDX = np.sort(np.random.default_rng(0).choice(np.arange(len(TEST["inputs"])), size=POOL_N, replace=False))


def pool_batch(idx, copies=1):
    """The test puzzles `idx`, repeated `copies` times, as one padded batch of the test loader."""
    idx = np.tile(np.asarray(idx, dtype=np.int64), copies)
    pid = np.searchsorted(TEST["puzzle_indices"], idx, side="right") - 1
    rows = {"inputs": np.asarray(TEST["inputs"][idx]), "labels": np.asarray(TEST["labels"][idx]),
            "puzzle_identifiers": np.asarray(TEST["puzzle_identifiers"][pid])}
    return {k: v.to(DEVICE) for k, v in test_set._collate_batch(rows).items()}


# Every cached file must reproduce, bit for bit, the halting logits of this standard pass on the first batch.
ref_batch = pool_batch(POOL_IDX[:GLOBAL_BATCH_SIZE])
ref_carry, ref_pred, ref_q = run_block(ref_batch)
REF_Q0 = ref_q[:min(POOL_N, GLOBAL_BATCH_SIZE)].cpu().numpy()



def cached_pass(tag, arm, seed):
    """The pass of `arm` for `seed`, loaded from the cache or run and saved."""
    path = os.path.join(CACHE_DIR, "TRM_pool_policy_%s_seed%d.npz" % (tag, seed))
    if not os.path.exists(path):
        res = run_pass(arm, seed)
        arrays = {"ok0": res["ok0"].astype(np.uint8), "q0": res["q0"], "pred0": res["pred0"].astype(np.int16),
                  arm + "__ok": res["ok"].astype(np.uint8), arm + "__act": res["act"].astype(np.uint8)}
        np.savez_compressed(path, pool_idx=POOL_IDX, **arrays)
    f = np.load(path)
    assert np.array_equal(f["pool_idx"], POOL_IDX) and np.array_equal(f["q0"][:len(REF_Q0)], REF_Q0), \
        "%s comes from other puzzles or another numerical setup (GPU, PyTorch): delete it to recompute it" % path
    return dict(ok0=f["ok0"].astype(bool), q0=f["q0"], pred0=f["pred0"],
                ok=f[arm + "__ok"].astype(bool), act=f[arm + "__act"].astype(bool))

In [ ]:
# Self-test: a restart that resamples nothing replays the standard pass bit for bit.
active = correct(ref_pred, ref_batch)[1] & (ref_q < Q_THR)
carry = clone(ref_carry)
retry(carry, torch.zeros_like(ref_batch["inputs"], dtype=torch.bool), active, "restart", torch.Generator(device=DEVICE))
_, pred, q = run_block(ref_batch, carry)
assert (pred[active] == ref_pred[active]).all() and (q[active] == ref_q[active]).all()

## GAR, continue instead of restart, TRM and PTRM

TRM is deterministic, so its pass runs once and serves the three seeds. Under the harness, each retry of PTRM is one
of its cached rollouts of D = 16 steps, read in order.

In [ ]:
PTRM_K, PTRM_SIGMA = 100, 0.3

GAR = [cached_pass("RST%03d" % ROUNDS, "restart", s) for s in SEEDS]
CONT = [cached_pass("R%03d" % ROUNDS, "randvec", s) for s in SEEDS]
TRM = [cached_pass("R%03dnone" % ROUNDS, "none", 0)] * len(SEEDS)
ok0, q0 = GAR[0]["ok0"], GAR[0]["q0"]
assert all(np.array_equal(p["q0"], q0) and np.array_equal(p["pred0"], GAR[0]["pred0"]) for p in GAR + CONT + TRM)

PTRM = []
for s in SEEDS:
    path = os.path.join(CACHE_DIR, "TRM__K%d_D16_s%.2f__seed%d.npz" % (PTRM_K, PTRM_SIGMA, s))
    assert os.path.exists(path), "%s is missing: run SudokuExtreme_continue_and_PTRM first" % path
    f = np.load(path)
    assert np.array_equal(f["pool_idx"], POOL_IDX) and np.array_equal(f["ref_q0"], REF_Q0), \
        "%s comes from other puzzles or another numerical setup (GPU, PyTorch)" % path
    PTRM.append((f["ok"].astype(bool), f["q"]))


def ptrm_harness(ok, q, k):
    """PTRM under the harness: on a re-opened puzzle, the rollouts are read in order until one has q >= Q_THR, at
    most k of them, and the fallback returns the grid of highest q among the standard pass and the rollouts read.
    Returns the correctness per puzzle, the cost in blocks per puzzle (one block per rollout of D = T steps) and the
    number of re-opened puzzles still searching before the k-th rollout."""
    ok, q = ok[:k], q[:k]
    n = ok.shape[1]
    hit = q >= Q_THR
    last = np.where(hit.any(0), hit.argmax(0), k - 1)
    read = np.concatenate([np.ones((1, n), bool), np.arange(k)[:, None] <= last[None, :]])
    best = np.where(read, np.concatenate([q0[None, :], q]), -np.inf).argmax(0)
    gate = q0 < Q_THR
    return np.where(gate, np.concatenate([ok0[None, :], ok])[best, np.arange(n)], ok0), \
        1.0 + float(((last + 1) * gate).sum()) / n, (gate & ~hit[:k - 1].any(0)).sum()

## Results

After `r` retries: accuracy (%) and cost (blocks per puzzle), as mean and standard deviation over the seeds, and the
fraction (%) of re-opened puzzles still searching before retry `r`. The differences are paired by puzzle and
averaged over seeds, with a 95% bootstrap interval over puzzles (B = 10,000, the seeds of a puzzle kept together).

In [ ]:
R_LIST = [r for r in (1, 2, 5, 10, 16, 25, 32, 48, 50, 64) if r <= ROUNDS]
BOOT = np.random.default_rng(0).integers(0, POOL_N, size=(10000, POOL_N))
N_FAIL, N_OPEN = (~ok0).sum(), (q0 < Q_THR).sum()
METHODS = {"GAR": GAR, "continue instead of restart": CONT, "TRM": TRM}


def results(r):
    """Per method, one entry per seed: the correctness of each puzzle after r retries, the cost, and the number of
    re-opened puzzles still searching before retry r."""
    out = {name: ([p["ok"][:, r] for p in passes], [1.0 + p["act"][:, :r].sum() / POOL_N for p in passes],
                  [p["act"][:, r - 1].sum() for p in passes])
           for name, passes in METHODS.items()}
    out["PTRM"] = tuple(zip(*(ptrm_harness(ok, q, r) for ok, q in PTRM)))
    return out


def paired(a, b):
    """a - b in accuracy points, with a and b lists (one per seed) of per-puzzle correctness."""
    d = np.mean([x.astype(np.float64) for x in a], 0) - np.mean([x.astype(np.float64) for x in b], 0)
    lo, hi = np.percentile(d[BOOT].mean(1), [2.5, 97.5])
    return {"diff": 100 * d.mean(), "low": 100 * lo, "high": 100 * hi}


rows = {}
for r in R_LIST:
    res, row = results(r), {}
    for name, (ok, cost, still) in res.items():
        acc = [100 * v.mean() for v in ok]
        row.update({(name, "accuracy"): np.mean(acc), (name, "sd"): np.std(acc, ddof=1), (name, "cost"): np.mean(cost),
                    (name, "open"): 100 * np.mean(still) / N_OPEN})
    row.update({("GAR - PTRM", k): v for k, v in paired(res["GAR"][0], res["PTRM"][0]).items()})
    row.update({("continue - GAR", k): v
                for k, v in paired(res["continue instead of restart"][0], res["GAR"][0]).items()})
    rows[r] = row
TABLE = pd.DataFrame.from_dict(rows, orient="index").rename_axis("r")
print("Standard pass: %.2f%% of %d puzzles, %d failures, %d re-opened by the gate."
      % (100 * ok0.mean(), POOL_N, N_FAIL, N_OPEN))
TABLE

At 16 and 64 retries, the standard-pass failures recovered and the standard-pass successes lost (means over seeds).

In [ ]:
rows = {}
for r in (16, ROUNDS):
    for name, (ok, _, _) in results(r).items():
        rec = [(~ok0 & v).sum() for v in ok]
        rows[(r, name)] = {"recovered": np.mean(rec), "sd": np.std(rec, ddof=1),
                           "recovered (%)": 100 * np.mean(rec) / N_FAIL, "sd (%)": 100 * np.std(rec, ddof=1) / N_FAIL,
                           "lost": np.mean([(ok0 & ~v).sum() for v in ok])}
RECOVERED = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["r", "method"])
RECOVERED

Puzzles solved by only one of GAR and its continuation variant, per seed.

In [ ]:
SOLVED_BY_ONE = pd.DataFrame({(r, s): {"continue only": (CONT[i]["ok"][:, r] & ~GAR[i]["ok"][:, r]).sum(),
                                       "GAR only": (GAR[i]["ok"][:, r] & ~CONT[i]["ok"][:, r]).sum()}
                              for r in (16, ROUNDS) for i, s in enumerate(SEEDS)}).rename_axis(columns=["r", "seed"])
SOLVED_BY_ONE